# Alpha Research — Homework 3
**Jaskaran Kalra**

### Objective
Expand on Homework 2 by re-evaluating features and targets, adding stock-specific signals, moving beyond OLS, and computing portfolio analytics including Sharpe ratio.

### Sample Period
January 2010 — May 2026

### Universe
S&P 500 constituents as of January 1, 2010 (point-in-time)

In [7]:
import pandas as pd
import numpy as np
import os

DATA = '../../seminar_02/homework/data/'

# ── Load final panel from HW2 ─────────────────────────────────────────
panel = pd.read_csv(DATA + 'panel_clean.csv', parse_dates=['date'])

# ── Load prices to compute new stock-specific signals ─────────────────
prices_clean = pd.read_csv(DATA + 'sp500_prices_clean.csv', index_col=0, parse_dates=True)

print(f'Panel shape:  {panel.shape}')
print(f'Prices shape: {prices_clean.shape}')
print(f'\nPanel columns: {list(panel.columns)}')

Panel shape:  (1259364, 11)
Prices shape: (4117, 334)

Panel columns: ['date', 'ticker', 'forward_return', 'momentum', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE']


In [8]:
# ── Short-term reversal: 21-day return as of day T ────────────────────
# Stocks that went up last month tend to revert the following month
# Signal: last 21-day simple return (negative expected coefficient)
reversal = prices_clean.pct_change(21)

# Melt to long format to merge into panel
reversal_long = reversal.stack().reset_index()
reversal_long.columns = ['date', 'ticker', 'reversal']

# Merge into panel
panel = panel.merge(reversal_long, on=['date', 'ticker'], how='left')

print(f'Panel shape after adding reversal: {panel.shape}')
print(f'Reversal missing values: {panel["reversal"].isnull().sum()}')
print(f'\nReversal descriptive stats:\n{panel["reversal"].describe()}')

Panel shape after adding reversal: (1259364, 12)
Reversal missing values: 0

Reversal descriptive stats:
count    1.259364e+06
mean     1.202429e-02
std      8.822266e-02
min     -8.482394e-01
25%     -3.328711e-02
50%      1.226148e-02
75%      5.602447e-02
max      2.411985e+00
Name: reversal, dtype: float64


In [55]:
def compute_analytics(X, y, dates_col, model, model_name='Model'):
    # Get predictions
    if hasattr(model, 'coef_') or hasattr(model, 'params'):
        # statsmodels OLS
        try:
            pred = model.predict(sm.add_constant(X))
        except:
            pred = model.predict(X)
    else:
        pred = model.predict(X)

    df = pd.DataFrame({
        'date'           : dates_col.values,
        'forward_return' : y.values,
        'predicted'      : pred
    })

    # Long-short return each period
    monthly_returns = []
    for date, group in df.groupby('date'):
        if len(group) < 10:
            continue
        q20 = group['predicted'].quantile(0.2)
        q80 = group['predicted'].quantile(0.8)
        long_ret  = group[group['predicted'] >= q80]['forward_return'].mean()
        short_ret = group[group['predicted'] <= q20]['forward_return'].mean()
        monthly_returns.append(long_ret - short_ret)

    monthly_returns = pd.Series(monthly_returns)

    # Metrics
    sharpe        = monthly_returns.mean() / monthly_returns.std() * np.sqrt(12)
    annualized    = monthly_returns.mean() * 12
    win_rate      = (monthly_returns > 0).mean()
    cumulative    = (1 + monthly_returns).prod() - 1
    drawdown      = (monthly_returns.cumsum() - monthly_returns.cumsum().cummax()).min()

    print(f'── {model_name} ──────────────────────────')
    print(f'Sharpe ratio:      {sharpe:.4f}')
    print(f'Annualized return: {annualized*100:.2f}%')
    print(f'Total return:      {cumulative*100:.2f}%')
    print(f'Win rate:          {win_rate*100:.1f}%')
    print(f'Max drawdown:      {drawdown*100:.2f}%')
    print()

    return monthly_returns

In [9]:
corrupted_check = panel[panel['reversal'] > 5].groupby('ticker')['reversal'].max().sort_values(ascending=False)
print(corrupted_check)

Series([], Name: reversal, dtype: float64)


In [10]:
from scipy.stats import mstats

# ── Winsorize reversal ────────────────────────────────────────────────
panel['reversal'] = mstats.winsorize(panel['reversal'], limits=[0.01, 0.01])

print(f'Reversal range after winsorization: {panel["reversal"].min():.4f} to {panel["reversal"].max():.4f}')

Reversal range after winsorization: -0.2219 to 0.2546


In [11]:
# ── Standardize reversal ──────────────────────────────────────────────
mean = panel['reversal'].mean()
std  = panel['reversal'].std()
panel['reversal'] = (panel['reversal'] - mean) / std

print(f'Reversal mean: {panel["reversal"].mean():.6f}')
print(f'Reversal std:  {panel["reversal"].std():.6f}')

Reversal mean: 0.000000
Reversal std:  1.000000


In [12]:
import statsmodels.api as sm
from scipy.stats import mstats

# ── Apply same transformations as HW2 ────────────────────────────────
panel = panel.copy()
panel['forward_return'] = np.log(1 + panel['forward_return'])

for col in ['forward_return', 'momentum']:
    panel[col] = mstats.winsorize(panel[col], limits=[0.01, 0.01])

features = ['momentum', 'reversal', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE']

for col in features:
    mean = panel[col].mean()
    std  = panel[col].std()
    panel[col] = (panel[col] - mean) / std

# ── Subsample to non-overlapping observations ─────────────────────────
panel_monthly = (
    panel.groupby('ticker', group_keys=False)
    .apply(lambda x: x.iloc[::21])
    .reset_index(drop=True)
)

# ── Run OLS ───────────────────────────────────────────────────────────
X = sm.add_constant(panel_monthly[features])
y = panel_monthly['forward_return']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         forward_return   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.023
Method:                 Least Squares   F-statistic:                     162.0
Date:                Wed, 03 Jun 2026   Prob (F-statistic):          1.06e-304
Time:                        14:08:20   Log-Likelihood:                 68209.
No. Observations:               60281   AIC:                        -1.364e+05
Df Residuals:                   60271   BIC:                        -1.363e+05
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0091      0.000     27.927      0.0

### OLS Results — Adding Short-Term Reversal

Compared to the HW2 baseline (monthly non-overlapping panel, same universe):

| Metric | HW2 Baseline | HW3 + Reversal | Change |
|---|---|---|---|
| R² | 2.2% | 2.4% | +0.2pp |
| Durbin-Watson | 2.078 | 2.004 | ✓ clean |
| Kurtosis | 4.24 | 4.24 | unchanged |
| Observations | 60,281 | 60,281 | unchanged |

**Reversal coefficient: -0.0032 (t = -9.59, p = 0.000)**

The reversal signal is highly significant with the expected negative sign — stocks that outperformed over the last 21 days tend to underperform over the next 21 days. This is consistent with the short-term reversal effect documented by Jegadeesh (1990), driven by temporary price pressure from liquidity-motivated trading.

Adding reversal also slightly reduced the momentum coefficient (0.0014 → 0.0012), which is expected since the two signals are related — momentum captures 12-month drift while reversal captures the most recent month's mean reversion.

In [13]:
# ── Realized volatility: 21-day rolling std of daily returns ──────────
# Low-volatility anomaly: high vol stocks tend to underperform
# Expected coefficient: negative
volatility = prices_clean.pct_change().rolling(21).std()

# Melt to long format
vol_long = volatility.stack().reset_index()
vol_long.columns = ['date', 'ticker', 'volatility']

# Merge into panel
panel = panel.merge(vol_long, on=['date', 'ticker'], how='left')

print(f'Panel shape after adding volatility: {panel.shape}')
print(f'Volatility missing values: {panel["volatility"].isnull().sum()}')
print(f'\nVolatility descriptive stats:\n{panel["volatility"].describe()}')

Panel shape after adding volatility: (1259364, 13)
Volatility missing values: 152

Volatility descriptive stats:
count    1.259212e+06
mean     1.715966e-02
std      1.072538e-02
min      0.000000e+00
25%      1.071340e-02
50%      1.444683e-02
75%      2.010378e-02
max      2.246502e-01
Name: volatility, dtype: float64


In [15]:
panel = panel.dropna(subset=['volatility'])
print(f'Panel shape after dropping volatility NaNs: {panel.shape}')

Panel shape after dropping volatility NaNs: (1259212, 13)


In [16]:
# ── Winsorize and standardize volatility ─────────────────────────────
panel['volatility'] = mstats.winsorize(panel['volatility'], limits=[0.01, 0.01])

mean = panel['volatility'].mean()
std  = panel['volatility'].std()
panel['volatility'] = (panel['volatility'] - mean) / std

print(f'Volatility mean: {panel["volatility"].mean():.6f}')
print(f'Volatility std:  {panel["volatility"].std():.6f}')

Volatility mean: 0.000000
Volatility std:  1.000000


In [17]:
# ── OLS with reversal + volatility ────────────────────────────────────
features = ['momentum', 'reversal', 'volatility', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE']

# Standardize volatility
mean = panel['volatility'].mean()
std  = panel['volatility'].std()
panel['volatility'] = (panel['volatility'] - mean) / std

# Subsample to non-overlapping observations
panel_monthly = (
    panel.groupby('ticker', group_keys=False)
    .apply(lambda x: x.iloc[::21])
    .reset_index(drop=True)
)

X = sm.add_constant(panel_monthly[features])
y = panel_monthly['forward_return']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         forward_return   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.024
Method:                 Least Squares   F-statistic:                     147.4
Date:                Wed, 03 Jun 2026   Prob (F-statistic):          6.18e-307
Time:                        14:36:38   Log-Likelihood:                 68222.
No. Observations:               60274   AIC:                        -1.364e+05
Df Residuals:                   60263   BIC:                        -1.363e+05
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0091      0.000     27.961      0.0

### OLS Results — Adding Volatility

| Metric | HW2 Baseline | + Reversal | + Volatility |
|---|---|---|---|
| R² | 2.2% | 2.4% | 2.4% |
| Durbin-Watson | 2.078 | 2.004 | 2.003 |
| Kurtosis | 4.24 | 4.24 | 4.23 |
| Features | 8 | 9 | 10 |

**Volatility coefficient: -0.0012 (t = -3.09, p = 0.002)**

Volatility is statistically significant with the expected negative sign — consistent with the low-volatility anomaly. However its economic contribution is modest: R² did not improve, suggesting volatility overlaps with existing features, particularly VIX which already captures market-wide fear. Stock-level volatility adds incremental signal but less than reversal.

**Momentum has weakened** progressively as new features are added (0.0014 → 0.0012 → 0.0010), indicating that reversal and volatility are capturing part of what momentum was previously proxying for.

In [18]:
# ── 52-week high ratio: price / max price over last 252 days ──────────
# Stocks near their 52-week high tend to outperform (George & Hwang 2004)
# Expected coefficient: positive
high_52w = prices_clean.rolling(252).max()
ratio_52w = prices_clean / high_52w

# Melt to long format
ratio_long = ratio_52w.stack().reset_index()
ratio_long.columns = ['date', 'ticker', 'ratio_52w']

# Merge into panel
panel = panel.merge(ratio_long, on=['date', 'ticker'], how='left')

print(f'Panel shape after adding 52w high ratio: {panel.shape}')
print(f'Missing values: {panel["ratio_52w"].isnull().sum()}')
print(f'\nDescriptive stats:\n{panel["ratio_52w"].describe()}')

Panel shape after adding 52w high ratio: (1259212, 14)
Missing values: 1262

Descriptive stats:
count    1.257950e+06
mean     8.808786e-01
std      1.261641e-01
min      5.730994e-02
25%      8.300064e-01
50%      9.211821e-01
75%      9.738948e-01
max      1.000000e+00
Name: ratio_52w, dtype: float64


In [19]:
# ── Drop NaNs and standardize ─────────────────────────────────────────
panel = panel.dropna(subset=['ratio_52w'])

mean = panel['ratio_52w'].mean()
std  = panel['ratio_52w'].std()
panel['ratio_52w'] = (panel['ratio_52w'] - mean) / std

print(f'Panel shape: {panel.shape}')
print(f'ratio_52w mean: {panel["ratio_52w"].mean():.6f}')
print(f'ratio_52w std:  {panel["ratio_52w"].std():.6f}')

Panel shape: (1257950, 14)
ratio_52w mean: 0.000000
ratio_52w std:  1.000000


In [20]:
# ── OLS with reversal + volatility + 52-week high ratio ───────────────
features = ['momentum', 'reversal', 'volatility', 'ratio_52w', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE']

panel_monthly = (
    panel.groupby('ticker', group_keys=False)
    .apply(lambda x: x.iloc[::21])
    .reset_index(drop=True)
)

X = sm.add_constant(panel_monthly[features])
y = panel_monthly['forward_return']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         forward_return   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.024
Method:                 Least Squares   F-statistic:                     135.4
Date:                Wed, 03 Jun 2026   Prob (F-statistic):          4.05e-309
Time:                        14:43:12   Log-Likelihood:                 68203.
No. Observations:               60213   AIC:                        -1.364e+05
Df Residuals:                   60201   BIC:                        -1.363e+05
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0090      0.000     27.873      0.0

### Feature Selection — Dropping Volatility and 52-Week High Ratio

**Volatility** (t = -3.09): statistically significant with the correct negative sign, but added no improvement to R². Stock-level volatility overlaps heavily with VIX which already captures market-wide fear. Dropped as it adds complexity without incremental predictive value.

**52-week high ratio** (t = -3.69): significant but with the **wrong sign** — negative instead of the expected positive. In our sample the reversal effect dominates the momentum anchoring effect at the 21-day horizon. Also caused multicollinearity with momentum (Cond. No. rose from 2.51 to 3.95) and added no R² improvement. Dropped.

**Retained features:** momentum + reversal + original macro features (VIX, Mkt_RF, SMB, HML, FEDFUNDS, CPI_YoY, UNRATE).

In [21]:
# ── Drop features that did not add value ─────────────────────────────
panel = panel.drop(columns=['volatility', 'ratio_52w'])
print(f'Final panel shape: {panel.shape}')
print(f'Columns: {list(panel.columns)}')

Final panel shape: (1257950, 12)
Columns: ['date', 'ticker', 'forward_return', 'momentum', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE', 'reversal']


## Fama-MacBeth Regression

Pooled OLS treats all 60,281 observations as independent, but stocks in the same month are correlated — they all react to the same market environment. This inflates t-statistics and makes signals look more significant than they really are.

Fama-MacBeth (1973) addresses this by running one cross-sectional regression per month across all stocks, then averaging the monthly coefficients. The t-statistics are computed from how much those coefficients vary over time — measuring signal consistency rather than sample size.

In [22]:
# ── Fama-MacBeth Regression ───────────────────────────────────────────
# Step 1: run cross-sectional OLS for each month
# Step 2: average coefficients across months, compute t-stats from time-series variation

features = ['momentum', 'reversal', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE']

# Use the monthly subsampled panel
panel_monthly = (
    panel.groupby('ticker', group_keys=False)
    .apply(lambda x: x.iloc[::21])
    .reset_index(drop=True)
)

# Step 1: cross-sectional regression for each date
monthly_coefs = []

for date, group in panel_monthly.groupby('date'):
    if len(group) < 10:  # skip months with too few stocks
        continue
    X = sm.add_constant(group[features])
    y = group['forward_return']
    result = sm.OLS(y, X).fit()
    monthly_coefs.append(result.params)

coefs = pd.DataFrame(monthly_coefs)

# Step 2: average across months and compute t-stats
means = coefs.mean()
stds  = coefs.std()
tstat = means / (stds / np.sqrt(len(coefs)))

fm_results = pd.DataFrame({
    'mean_coef': means,
    'std':       stds,
    't_stat':    tstat
})

print(f'Number of monthly regressions: {len(coefs)}')
print(f'\nFama-MacBeth Results:')
print(fm_results.round(6))

Number of monthly regressions: 183

Fama-MacBeth Results:
          mean_coef       std    t_stat
momentum   0.001768  0.015661  1.526945
reversal  -0.000435  0.013630 -0.432137
VIX        0.000012  0.005338  0.029792
Mkt_RF     0.000569  0.005212  1.475796
SMB        0.000771  0.007934  1.314573
HML        0.000438  0.005195  1.139509
FEDFUNDS  -0.000932  0.006693 -1.882891
CPI_YoY   -0.001234  0.005094 -3.278463
UNRATE     0.000847  0.007829  1.463762


### Fama-MacBeth Results

| Feature | t-stat (Pooled OLS) | t-stat (Fama-MacBeth) |
|---|---|---|
| momentum | 3.70 | 1.53 |
| reversal | -9.59 | -0.43 |
| VIX | 22.13 | 0.03 |
| Mkt_RF | -8.33 | 1.48 |
| SMB | 18.08 | 1.31 |
| HML | 7.22 | 1.14 |
| FEDFUNDS | 10.28 | -1.88 |
| CPI_YoY | -17.65 | **-3.28** |
| UNRATE | 5.56 | 1.46 |

The contrast is stark — pooled OLS showed nearly every feature as highly significant, while Fama-MacBeth shows only **CPI_YoY** clears the |t| > 2 threshold. This confirms that pooled OLS was inflating t-statistics by ignoring cross-sectional correlation between stocks.

The low t-stats indicate that most signals are **inconsistent over time** — they work in some months but reverse in others. This motivates moving beyond linear models to capture non-linear relationships and interactions between features.

In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

features = ['momentum', 'reversal', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE']

# ── Train/test split by time ──────────────────────────────────────────
# Use first 80% of dates for training, last 20% for testing
# Important: never shuffle — future data must not leak into training
panel_monthly = (
    panel.groupby('ticker', group_keys=False)
    .apply(lambda x: x.iloc[::21])
    .reset_index(drop=True)
)

dates = sorted(panel_monthly['date'].unique())
cutoff = dates[int(len(dates) * 0.8)]

train = panel_monthly[panel_monthly['date'] <= cutoff]
test  = panel_monthly[panel_monthly['date'] > cutoff]

X_train = train[features]
y_train = train['forward_return']
X_test  = test[features]
y_test  = test['forward_return']

print(f'Train: {train["date"].min().date()} to {train["date"].max().date()} ({len(train)} rows)')
print(f'Test:  {test["date"].min().date()} to {test["date"].max().date()} ({len(test)} rows)')

Train: 2011-02-02 to 2023-09-11 (50084 rows)
Test:  2023-09-21 to 2026-04-16 (10129 rows)


In [27]:
# ── Train Random Forest ───────────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=3,        # shallower trees
    min_samples_leaf=200,  # each leaf needs more samples
    max_features=0.5,   # only consider half the features at each split
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────
y_pred_train = rf.predict(X_train)
y_pred_test  = rf.predict(X_test)

print(f'R² train: {r2_score(y_train, y_pred_train):.4f}')
print(f'R² test:  {r2_score(y_test, y_pred_test):.4f}')

# ── Feature importances ───────────────────────────────────────────────
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(f'\nFeature importances:\n{importances.round(4)}')

R² train: 0.1212
R² test:  -0.0264

Feature importances:
Mkt_RF      0.3405
CPI_YoY     0.2549
VIX         0.1591
SMB         0.1457
UNRATE      0.0422
HML         0.0309
FEDFUNDS    0.0139
momentum    0.0109
reversal    0.0020
dtype: float64


In [28]:
# ── 3-month and 6-month momentum ─────────────────────────────────────
mom_3m = np.log(prices_clean).diff(63).shift(21)
mom_6m = np.log(prices_clean).diff(126).shift(21)

# Melt to long format
mom_3m_long = mom_3m.stack().reset_index()
mom_3m_long.columns = ['date', 'ticker', 'mom_3m']

mom_6m_long = mom_6m.stack().reset_index()
mom_6m_long.columns = ['date', 'ticker', 'mom_6m']

# Merge into panel
panel = panel.merge(mom_3m_long, on=['date', 'ticker'], how='left')
panel = panel.merge(mom_6m_long, on=['date', 'ticker'], how='left')

print(f'Panel shape: {panel.shape}')
print(f'mom_3m missing: {panel["mom_3m"].isnull().sum()}')
print(f'mom_6m missing: {panel["mom_6m"].isnull().sum()}')

Panel shape: (1257950, 14)
mom_3m missing: 0
mom_6m missing: 0


In [29]:
# ── Drop NaNs, winsorize and standardize ─────────────────────────────
panel = panel.dropna(subset=['mom_3m', 'mom_6m'])

for col in ['mom_3m', 'mom_6m']:
    panel[col] = mstats.winsorize(panel[col], limits=[0.01, 0.01])
    mean = panel[col].mean()
    std  = panel[col].std()
    panel[col] = (panel[col] - mean) / std

print(f'Panel shape: {panel.shape}')
print(f'mom_3m — mean: {panel["mom_3m"].mean():.4f}, std: {panel["mom_3m"].std():.4f}')
print(f'mom_6m — mean: {panel["mom_6m"].mean():.4f}, std: {panel["mom_6m"].std():.4f}')

Panel shape: (1257950, 14)
mom_3m — mean: 0.0000, std: 1.0000
mom_6m — mean: -0.0000, std: 1.0000


In [30]:
# ── Retrain Random Forest with additional momentum features ───────────
features = ['momentum', 'mom_3m', 'mom_6m', 'reversal', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE']

panel_monthly = (
    panel.groupby('ticker', group_keys=False)
    .apply(lambda x: x.iloc[::21])
    .reset_index(drop=True)
)

dates = sorted(panel_monthly['date'].unique())
cutoff = dates[int(len(dates) * 0.8)]

train = panel_monthly[panel_monthly['date'] <= cutoff]
test  = panel_monthly[panel_monthly['date'] > cutoff]

X_train = train[features]
y_train = train['forward_return']
X_test  = test[features]
y_test  = test['forward_return']

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=3,
    min_samples_leaf=200,
    max_features=0.5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print(f'R² train: {r2_score(y_train, rf.predict(X_train)):.4f}')
print(f'R² test:  {r2_score(y_test, rf.predict(X_test)):.4f}')

importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(f'\nFeature importances:\n{importances.round(4)}')

R² train: 0.1171
R² test:  -0.0208

Feature importances:
Mkt_RF      0.3404
CPI_YoY     0.2492
VIX         0.1495
SMB         0.1260
HML         0.0458
UNRATE      0.0410
FEDFUNDS    0.0291
momentum    0.0135
reversal    0.0028
mom_6m      0.0020
mom_3m      0.0008
dtype: float64


In [31]:
from sklearn.linear_model import RidgeCV, LassoCV

features = ['momentum', 'mom_3m', 'mom_6m', 'reversal', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE']

# ── Ridge Regression ──────────────────────────────────────────────────
ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0], cv=5)
ridge.fit(X_train, y_train)

print(f'Ridge best alpha: {ridge.alpha_}')
print(f'Ridge R² train:  {r2_score(y_train, ridge.predict(X_train)):.4f}')
print(f'Ridge R² test:   {r2_score(y_test, ridge.predict(X_test)):.4f}')
print(f'\nRidge coefficients:')
print(pd.Series(ridge.coef_, index=features).sort_values(ascending=False).round(6))

print()

# ── Lasso Regression ──────────────────────────────────────────────────
lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1], cv=5, max_iter=5000)
lasso.fit(X_train, y_train)

print(f'Lasso best alpha: {lasso.alpha_}')
print(f'Lasso R² train:  {r2_score(y_train, lasso.predict(X_train)):.4f}')
print(f'Lasso R² test:   {r2_score(y_test, lasso.predict(X_test)):.4f}')
print(f'\nLasso coefficients:')
print(pd.Series(lasso.coef_, index=features).sort_values(ascending=False).round(6))

Ridge best alpha: 10.0
Ridge R² train:  0.0310
Ridge R² test:   -0.0168

Ridge coefficients:
VIX         0.009638
SMB         0.008202
HML         0.003014
FEDFUNDS    0.002188
UNRATE      0.001621
mom_6m      0.000689
momentum    0.000558
mom_3m     -0.000194
Mkt_RF     -0.002833
reversal   -0.004130
CPI_YoY    -0.006394
dtype: float64

Lasso best alpha: 0.0001
Lasso R² train:  0.0310
Lasso R² test:   -0.0177

Lasso coefficients:
VIX         0.009470
SMB         0.008025
HML         0.002894
FEDFUNDS    0.001636
UNRATE      0.001458
mom_6m      0.000519
momentum    0.000372
mom_3m     -0.000000
Mkt_RF     -0.002660
reversal   -0.004039
CPI_YoY    -0.006184
dtype: float64


In [32]:
# ── OLS on train/test split ───────────────────────────────────────────
features_ols = ['momentum', 'reversal', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE']

X_train_ols = sm.add_constant(train[features_ols])
X_test_ols  = sm.add_constant(test[features_ols])

model_ols = sm.OLS(y_train, X_train_ols).fit()

r2_train_ols = r2_score(y_train, model_ols.predict(X_train_ols))
r2_test_ols  = r2_score(y_test,  model_ols.predict(X_test_ols))

print(f'OLS R² train: {r2_train_ols:.4f}')
print(f'OLS R² test:  {r2_test_ols:.4f}')

OLS R² train: 0.0310
OLS R² test:  -0.0169


In [37]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=50,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

print(f'XGBoost R² train: {r2_score(y_train, xgb.predict(X_train)):.4f}')
print(f'XGBoost R² test:  {r2_score(y_test, xgb.predict(X_test)):.4f}')

importances = pd.Series(xgb.feature_importances_, index=features).sort_values(ascending=False)
print(f'\nFeature importances:\n{importances.round(4)}')

XGBoost R² train: 0.2305
XGBoost R² test:  -0.0668

Feature importances:
Mkt_RF      0.2201
CPI_YoY     0.1518
VIX         0.1396
SMB         0.1260
HML         0.0875
UNRATE      0.0861
FEDFUNDS    0.0670
momentum    0.0487
reversal    0.0349
mom_3m      0.0221
mom_6m      0.0162
dtype: float32


## Model Comparison

### Train/Test Split
- **Train**: 2010 — ~2023 (80% of dates)
- **Test**: ~2023 — 2026 (20% of dates)

| Model | R² Train | R² Test | Notes |
|---|---|---|---|
| OLS (full sample) | 2.2% | — | In-sample only, no split |
| OLS | 3.1% | -1.7% | Linear, no regularization |
| Ridge | 3.1% | -1.7% | L2 regularization, best alpha = 10 |
| Lasso | 3.1% | -1.8% | L1 regularization, zeroed mom_3m |
| Random Forest | 11.7% | -2.1% | Non-linear, max_depth=3 |
| XGBoost | 23.1% | -6.7% | Gradient boosting, most overfit |

### Key Findings

**All models fail out-of-sample.** The 2022-2026 test period represents a distinct macro regime — aggressive Fed tightening, inflation shock, and rate hikes — that is fundamentally different from the 2010-2022 training period of quantitative easing and near-zero rates. Models trained on one regime do not generalise to the other.

**More complex models overfit more.** OLS, Ridge, and Lasso all produce identical train R² (~3.1%) and similar test R², confirming regularization adds little when the problem is regime change, not multicollinearity. Random Forest and XGBoost memorise training data better but generalise worse.

**Features are dominated by macro signals.** Mkt_RF, VIX, CPI_YoY and SMB account for ~70% of feature importance across all tree-based models. Stock-specific signals (momentum, reversal) collectively contribute less than 15%. This limits the model's ability to select individual stocks — it primarily learns market-wide patterns.

**Implication:** To build a robust stock selection model, more stock-specific features are needed (fundamentals, earnings surprises, analyst revisions) and models should be retrained frequently to adapt to changing regimes.

## Stock-Specific Features Only

Hypothesis: macro features (VIX, Mkt_RF, SMB, HML, FEDFUNDS, CPI_YoY, UNRATE) learned regime-specific relationships during 2010-2022 that broke down in the 2022-2026 test period. Removing them and keeping only stock-specific signals (momentum, mom_3m, mom_6m, reversal) should improve out-of-sample generalisation.

| Model | R² Train (all) | R² Test (all) | R² Train (stock only) | R² Test (stock only) |
|---|---|---|---|---|
| OLS | 3.1% | -1.7% | 0.4% | -0.5% |
| Ridge | 3.1% | -1.7% | 0.4% | -0.5% |
| Lasso | 3.1% | -1.8% | 0.4% | -0.5% |
| Random Forest | 11.7% | -2.1% | 0.7% | **-0.2%** |
| XGBoost | 23.1% | -6.7% | 1.7% | -0.5% |

**Removing macro features significantly improves out-of-sample performance across all models.** The test R² gap narrowed from -1.7% to -6.7% down to -0.2% to -0.5%. This confirms that macro variables were the primary driver of regime-specific overfitting.

However, train R² also collapsed — stock-specific price signals alone explain very little variance. The next step is to add more stock-specific features from alternative data sources to improve both in-sample fit and out-of-sample generalisation.

In [38]:
# ── Price acceleration: is momentum speeding up or slowing down? ──────
# Positive = momentum accelerating (3m stronger than 6m) → positive expected sign
panel['price_accel'] = panel['mom_3m'] - panel['mom_6m']

print(f'Price acceleration stats:\n{panel["price_accel"].describe()}')

Price acceleration stats:
count    1.257950e+06
mean     6.000879e-17
std      7.966204e-01
min     -5.970892e+00
25%     -4.633587e-01
50%     -5.133042e-02
75%      4.023400e-01
max      5.813727e+00
Name: price_accel, dtype: float64


In [39]:
# ── Rolling Beta: stock sensitivity to market moves ───────────────────
# Use SPY as market proxy
import yfinance as yf

spy = yf.download('SPY', start='2010-01-01', end='2026-05-16', auto_adjust=True)['Close']
spy_returns = spy.pct_change().squeeze()

# Compute rolling 63-day beta for each stock
stock_returns = prices_clean.pct_change()

def rolling_beta(stock_ret, market_ret, window=63):
    cov = stock_ret.rolling(window).cov(market_ret)
    var = market_ret.rolling(window).var()
    return cov / var

beta = stock_returns.apply(lambda col: rolling_beta(col, spy_returns))

# Melt and merge
beta_long = beta.stack().reset_index()
beta_long.columns = ['date', 'ticker', 'beta']

panel = panel.merge(beta_long, on=['date', 'ticker'], how='left')

print(f'Panel shape: {panel.shape}')
print(f'Beta missing: {panel["beta"].isnull().sum()}')
print(f'\nBeta stats:\n{panel["beta"].describe()}')

[*********************100%***********************]  1 of 1 completed


Panel shape: (1257950, 16)
Beta missing: 0

Beta stats:
count    1.257950e+06
mean     9.659694e-01
std      5.072714e-01
min     -4.199450e+00
25%      6.530956e-01
50%      9.522983e-01
75%      1.248390e+00
max      6.261519e+00
Name: beta, dtype: float64


In [40]:
# ── Winsorize and standardize beta ────────────────────────────────────
panel = panel.dropna(subset=['beta'])
panel['beta'] = mstats.winsorize(panel['beta'], limits=[0.01, 0.01])
mean = panel['beta'].mean()
std  = panel['beta'].std()
panel['beta'] = (panel['beta'] - mean) / std

print(f'Beta mean: {panel["beta"].mean():.4f}  std: {panel["beta"].std():.4f}')
print(f'Panel shape: {panel.shape}')

Beta mean: 0.0000  std: 1.0000
Panel shape: (1257950, 16)


In [44]:
# ── Fix: ensure beta_raw is dates × tickers ──────────────────────────
beta_raw = stock_returns.apply(lambda col: rolling_beta(col, spy_returns))

# If transposed, fix it
if beta_raw.index.dtype == 'O':  # tickers as index
    beta_raw = beta_raw.T

# ── Idiosyncratic volatility ──────────────────────────────────────────
total_var  = stock_returns.rolling(63).var()
market_var = spy_returns.rolling(63).var()

# Align market_var to total_var index
market_var = market_var.reindex(total_var.index)

idio_var = total_var.subtract(beta_raw.pow(2).multiply(market_var, axis=0))
idio_vol = np.sqrt(idio_var.clip(lower=0))

# Melt and merge
idio_long = idio_vol.stack().reset_index()
idio_long.columns = ['date', 'ticker', 'idio_vol']

panel = panel.merge(idio_long, on=['date', 'ticker'], how='left')

print(f'Panel shape: {panel.shape}')
print(f'Idio vol missing: {panel["idio_vol"].isnull().sum()}')
print(f'\nIdio vol stats:\n{panel["idio_vol"].describe()}')

Panel shape: (1257950, 19)
Idio vol missing: 0

Idio vol stats:
count    1.257950e+06
mean     1.426390e-02
std      7.910459e-03
min      1.708058e-03
25%      9.490487e-03
50%      1.224649e-02
75%      1.648653e-02
max      1.489572e-01
Name: idio_vol, dtype: float64


In [45]:
# ── Winsorize and standardize idio_vol ────────────────────────────────
panel['idio_vol'] = mstats.winsorize(panel['idio_vol'], limits=[0.01, 0.01])
mean = panel['idio_vol'].mean()
std  = panel['idio_vol'].std()
panel['idio_vol'] = (panel['idio_vol'] - mean) / std

print(f'idio_vol mean: {panel["idio_vol"].mean():.4f}  std: {panel["idio_vol"].std():.4f}')

idio_vol mean: 0.0000  std: 1.0000


In [46]:
# ── Winsorize and standardize idio_vol ────────────────────────────────
panel['idio_vol'] = mstats.winsorize(panel['idio_vol'], limits=[0.01, 0.01])
mean = panel['idio_vol'].mean()
std  = panel['idio_vol'].std()
panel['idio_vol'] = (panel['idio_vol'] - mean) / std

print(f'idio_vol mean: {panel["idio_vol"].mean():.4f}  std: {panel["idio_vol"].std():.4f}')

idio_vol mean: 0.0000  std: 1.0000


In [48]:
# ── All models with expanded stock-specific features ──────────────────
features_stock_v2 = ['momentum', 'mom_3m', 'mom_6m', 'reversal', 'price_accel', 'beta', 'idio_vol']

panel_monthly = (
    panel.groupby('ticker', group_keys=False)
    .apply(lambda x: x.iloc[::21])
    .reset_index(drop=True)
)

dates = sorted(panel_monthly['date'].unique())
cutoff = dates[int(len(dates) * 0.8)]

train = panel_monthly[panel_monthly['date'] <= cutoff]
test  = panel_monthly[panel_monthly['date'] > cutoff]

X_train_v2 = train[features_stock_v2]
X_test_v2  = test[features_stock_v2]

# ── OLS ───────────────────────────────────────────────────────────────
model_ols_v2 = sm.OLS(y_train, sm.add_constant(X_train_v2)).fit()
print(f'OLS       R² train: {r2_score(y_train, model_ols_v2.predict(sm.add_constant(X_train_v2))):.4f}  test: {r2_score(y_test, model_ols_v2.predict(sm.add_constant(X_test_v2))):.4f}')

# ── Ridge ─────────────────────────────────────────────────────────────
ridge_v2 = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0], cv=5)
ridge_v2.fit(X_train_v2, y_train)
print(f'Ridge     R² train: {r2_score(y_train, ridge_v2.predict(X_train_v2)):.4f}  test: {r2_score(y_test, ridge_v2.predict(X_test_v2)):.4f}')

# ── Lasso ─────────────────────────────────────────────────────────────
lasso_v2 = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1], cv=5, max_iter=5000)
lasso_v2.fit(X_train_v2, y_train)
print(f'Lasso     R² train: {r2_score(y_train, lasso_v2.predict(X_train_v2)):.4f}  test: {r2_score(y_test, lasso_v2.predict(X_test_v2)):.4f}')

# ── Random Forest ─────────────────────────────────────────────────────
rf_v2 = RandomForestRegressor(n_estimators=100, max_depth=3, min_samples_leaf=200, max_features=0.5, random_state=42, n_jobs=-1)
rf_v2.fit(X_train_v2, y_train)
print(f'RF        R² train: {r2_score(y_train, rf_v2.predict(X_train_v2)):.4f}  test: {r2_score(y_test, rf_v2.predict(X_test_v2)):.4f}')

# ── XGBoost ───────────────────────────────────────────────────────────
xgb_v2 = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, min_child_weight=50, random_state=42, n_jobs=-1)
xgb_v2.fit(X_train_v2, y_train)
print(f'XGBoost   R² train: {r2_score(y_train, xgb_v2.predict(X_train_v2)):.4f}  test: {r2_score(y_test, xgb_v2.predict(X_test_v2)):.4f}')

OLS       R² train: 0.0047  test: -0.0058
Ridge     R² train: 0.0047  test: -0.0058
Lasso     R² train: 0.0047  test: -0.0053
RF        R² train: 0.0091  test: -0.0042
XGBoost   R² train: 0.0217  test: -0.0075


In [49]:
# ── Long-short portfolio Sharpe ratio ────────────────────────────────
features_final = ['momentum', 'mom_3m', 'mom_6m', 'reversal']

# Use OLS predictions — best out-of-sample model
model_final = sm.OLS(y_train, sm.add_constant(X_train_s)).fit()

def compute_sharpe(X, y, dates_col, model):
    pred = model.predict(sm.add_constant(X))
    df = pd.DataFrame({
        'date'           : dates_col.values,
        'forward_return' : y.values,
        'predicted'      : pred
    })

    # Each month: long top 20%, short bottom 20%
    monthly_returns = []
    for date, group in df.groupby('date'):
        if len(group) < 10:
            continue
        q20 = group['predicted'].quantile(0.2)
        q80 = group['predicted'].quantile(0.8)
        long_ret  = group[group['predicted'] >= q80]['forward_return'].mean()
        short_ret = group[group['predicted'] <= q20]['forward_return'].mean()
        monthly_returns.append(long_ret - short_ret)

    monthly_returns = pd.Series(monthly_returns)
    sharpe = monthly_returns.mean() / monthly_returns.std() * np.sqrt(12)
    return sharpe, monthly_returns

sharpe_train, ret_train = compute_sharpe(X_train_s, y_train, train['date'], model_final)
sharpe_test,  ret_test  = compute_sharpe(X_test_s,  y_test,  test['date'],  model_final)

print(f'Sharpe ratio — train: {sharpe_train:.4f}')
print(f'Sharpe ratio — test:  {sharpe_test:.4f}')
print(f'\nMean monthly return — train: {ret_train.mean()*100:.2f}%')
print(f'Mean monthly return — test:  {ret_test.mean()*100:.2f}%')

Sharpe ratio — train: 0.3692
Sharpe ratio — test:  -0.2894

Mean monthly return — train: 0.40%
Mean monthly return — test:  -0.35%


## Summary & Findings

### 1. Feature Engineering

Starting from the HW2 baseline (momentum + macro features), we systematically added and evaluated stock-specific signals:

| Feature | t-stat | Added R²? | Decision |
|---|---|---|---|
| Short-term reversal | -9.59 | +0.2pp | ✅ Kept |
| Volatility | -3.09 | 0 | ❌ Dropped |
| 52-week high ratio | -3.69 (wrong sign) | 0 | ❌ Dropped |
| 3-month momentum | significant | marginal | ✅ Kept |
| 6-month momentum | significant | marginal | ✅ Kept |
| Price acceleration | derived | — | ✅ Kept |
| Beta | — | negative | ❌ Dropped |
| Idiosyncratic vol | — | negative | ❌ Dropped |

**Key insight:** Reversal was the strongest new signal. Volatility and 52-week high ratio added no incremental value over existing features.

---

### 2. Beyond OLS

| Model | R² Train | R² Test | Notes |
|---|---|---|---|
| OLS (full sample) | 2.2% | — | In-sample baseline |
| OLS | 3.1% | -1.7% | Linear, honest split |
| Ridge | 3.1% | -1.7% | Regularization didn't help |
| Lasso | 3.1% | -1.8% | Zeroed out mom_3m |
| Random Forest | 11.7% | -2.1% | Overfit |
| XGBoost | 23.1% | -6.7% | Most overfit |

**With stock-only features (momentum, mom_3m, mom_6m, reversal):**

| Model | R² Train | R² Test |
|---|---|---|
| OLS | 0.4% | -0.5% |
| Ridge | 0.4% | -0.5% |
| Lasso | 0.4% | -0.5% |
| Random Forest | 0.7% | **-0.2%** |
| XGBoost | 1.7% | -0.5% |

Removing macro features significantly improved out-of-sample generalisation. More complex models overfit more — linear models are more robust to regime changes.

---

### 3. Fama-MacBeth

Only **CPI_YoY** (t = -3.28) survived the Fama-MacBeth t-statistic threshold of |t| > 2. Pooled OLS was inflating t-statistics by ignoring cross-sectional correlation between stocks in the same month.

---

### 4. Portfolio Analytics

Using OLS with stock-specific features, constructing a long-short portfolio (long top 20%, short bottom 20% of predicted returns each month):

| Period | Sharpe Ratio | Mean Monthly Return |
|---|---|---|
| Train (2010–2023) | **0.37** | +0.40% |
| Test (2023–2026) | -0.29 | -0.35% |

The strategy generates a positive Sharpe ratio in-sample but breaks down in the 2022-2026 test period.

---

### 5. Root Cause: Regime Change

The 2022-2026 test period represents a fundamentally different macro regime — aggressive Fed tightening (0% → 5.25%), inflation shock (CPI peaked at 9%), and the end of quantitative easing. Models trained on the 2010-2022 low-rate environment do not generalise to this regime.

---

### 6. Limitations & Next Steps

- **Data:** Price-only features are insufficient for stock selection. Fundamentals (P/E, P/B, earnings growth), earnings surprises, and analyst revisions would significantly improve the model.
- **Regime adaptation:** Models should be retrained on a rolling basis (e.g. 3-year rolling window) rather than using a fixed training set.
- **Transaction costs:** The 0.40% monthly long-short return would be significantly eroded by transaction costs and market impact in practice.
- **Universe:** Survivorship bias from 26% failed yfinance downloads remains a limitation — a commercial database (CRSP) would eliminate this.

## Experiment: Reduced Training Window

**Hypothesis:** Training on 2010-2022 includes the pre-2018 era of near-zero rates and quantitative easing — a regime very different from 2022-2026. By training only on 2018-2022, we use a shorter but more recent and potentially more relevant window, closer in character to the test period.

**Risk:** Fewer training observations (~3,000 vs ~48,000) may lead to noisier coefficient estimates.

### Results — Reduced Training Window (2018-2022)

| Model | R² Train | R² Test | Sharpe Test |
|---|---|---|---|
| OLS (2010-2022) | 0.4% | -0.5% | -0.29 |
| OLS (2018-2022) | 1.1% | -1.3% | -0.08 |
| RF (2018-2022) | 2.3% | -0.6% | — |
| XGBoost (2018-2022) | 5.0% | -1.6% | — |

**Reducing the training window made R² worse but improved the Sharpe ratio** (-0.29 → -0.08). The shorter window produces noisier coefficient estimates (fewer observations), hurting R², but the model loses less money in the test period because it avoids learning the most regime-specific patterns from 2010-2018.

However all models still produce negative test R². The regime change from the low-rate 2018-2022 period to the aggressive tightening of 2022-2026 remains too large for price-based signals alone to bridge.

**Next experiment:** change the prediction target from raw returns to market-adjusted returns (stock return minus SPY return), which removes market-wide direction and focuses purely on stock selection.

In [52]:
# ── Market-adjusted return as target ──────────────────────────────────
# Instead of predicting raw return, predict excess return over SPY
# This removes market direction and focuses on stock selection

spy_returns_21d = spy.pct_change(21).shift(-21).squeeze()
spy_returns_21d.name = 'spy_21d'

# Merge SPY forward return into panel
panel_adj = panel.copy()
panel_adj = panel_adj.merge(spy_returns_21d.reset_index().rename(columns={'Date': 'date'}), on='date', how='left')
panel_adj['excess_return'] = panel_adj['forward_return'] - panel_adj['spy_21d']

print(f'Excess return stats:\n{panel_adj["excess_return"].describe()}')
print(f'Missing: {panel_adj["excess_return"].isnull().sum()}')

Excess return stats:
count    1.257950e+06
mean    -3.187785e-03
std      6.894109e-02
min     -4.288730e-01
25%     -4.082290e-02
50%     -1.616084e-03
75%      3.599371e-02
max      5.317636e-01
Name: excess_return, dtype: float64
Missing: 0


In [53]:
# ── All models with market-adjusted target ────────────────────────────
features_final = ['momentum', 'mom_3m', 'mom_6m', 'reversal']

panel_monthly_adj = (
    panel_adj.groupby('ticker', group_keys=False)
    .apply(lambda x: x.iloc[::21])
    .reset_index(drop=True)
)

dates = sorted(panel_monthly_adj['date'].unique())
cutoff = dates[int(len(dates) * 0.8)]

train_adj = panel_monthly_adj[panel_monthly_adj['date'] <= cutoff]
test_adj  = panel_monthly_adj[panel_monthly_adj['date'] > cutoff]

X_train_adj = train_adj[features_final]
X_test_adj  = test_adj[features_final]
y_train_adj = train_adj['excess_return']
y_test_adj  = test_adj['excess_return']

# ── OLS ───────────────────────────────────────────────────────────────
model_adj = sm.OLS(y_train_adj, sm.add_constant(X_train_adj)).fit()
print(f'OLS     R² train: {r2_score(y_train_adj, model_adj.predict(sm.add_constant(X_train_adj))):.4f}  test: {r2_score(y_test_adj, model_adj.predict(sm.add_constant(X_test_adj))):.4f}')

# ── RF ────────────────────────────────────────────────────────────────
rf_adj = RandomForestRegressor(n_estimators=100, max_depth=3, min_samples_leaf=200, max_features=0.5, random_state=42, n_jobs=-1)
rf_adj.fit(X_train_adj, y_train_adj)
print(f'RF      R² train: {r2_score(y_train_adj, rf_adj.predict(X_train_adj)):.4f}  test: {r2_score(y_test_adj, rf_adj.predict(X_test_adj)):.4f}')

# ── XGBoost ───────────────────────────────────────────────────────────
xgb_adj = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, min_child_weight=50, random_state=42, n_jobs=-1)
xgb_adj.fit(X_train_adj, y_train_adj)
print(f'XGBoost R² train: {r2_score(y_train_adj, xgb_adj.predict(X_train_adj)):.4f}  test: {r2_score(y_test_adj, xgb_adj.predict(X_test_adj)):.4f}')

# ── Sharpe ────────────────────────────────────────────────────────────
sharpe_adj, ret_adj = compute_sharpe(X_test_adj, y_test_adj, test_adj['date'], model_adj)
print(f'\nOLS Sharpe (test): {sharpe_adj:.4f}')
print(f'Mean monthly excess return (test): {ret_adj.mean()*100:.2f}%')

OLS     R² train: 0.0011  test: -0.0063
RF      R² train: 0.0044  test: -0.0051
XGBoost R² train: 0.0131  test: -0.0089

OLS Sharpe (test): -0.0391
Mean monthly excess return (test): -0.02%


In [54]:
# ── Ridge and Lasso on market-adjusted target ─────────────────────────
ridge_adj = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0], cv=5)
ridge_adj.fit(X_train_adj, y_train_adj)
print(f'Ridge   R² train: {r2_score(y_train_adj, ridge_adj.predict(X_train_adj)):.4f}  test: {r2_score(y_test_adj, ridge_adj.predict(X_test_adj)):.4f}')

lasso_adj = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1], cv=5, max_iter=5000)
lasso_adj.fit(X_train_adj, y_train_adj)
print(f'Lasso   R² train: {r2_score(y_train_adj, lasso_adj.predict(X_train_adj)):.4f}  test: {r2_score(y_test_adj, lasso_adj.predict(X_test_adj)):.4f}')

# ── Sharpe for each ───────────────────────────────────────────────────
for name, model, X_tr, X_te in [
    ('Ridge',   ridge_adj, X_train_adj, X_test_adj),
    ('Lasso',   lasso_adj, X_train_adj, X_test_adj),
    ('RF',      rf_adj,    X_train_adj, X_test_adj),
    ('XGBoost', xgb_adj,   X_train_adj, X_test_adj),
]:
    def predict(m, X):
        if hasattr(m, 'predict'):
            return m.predict(X)
    
    df_test = pd.DataFrame({
        'date'           : test_adj['date'].values,
        'forward_return' : y_test_adj.values,
        'predicted'      : model.predict(X_te)
    })
    monthly = []
    for date, group in df_test.groupby('date'):
        if len(group) < 10:
            continue
        q20 = group['predicted'].quantile(0.2)
        q80 = group['predicted'].quantile(0.8)
        monthly.append(group[group['predicted'] >= q80]['forward_return'].mean() - 
                       group[group['predicted'] <= q20]['forward_return'].mean())
    monthly = pd.Series(monthly)
    sharpe = monthly.mean() / monthly.std() * np.sqrt(12)
    print(f'{name:8s} Sharpe (test): {sharpe:.4f}  mean monthly: {monthly.mean()*100:.2f}%')

Ridge   R² train: 0.0011  test: -0.0063
Lasso   R² train: 0.0011  test: -0.0061
Ridge    Sharpe (test): -0.0278  mean monthly: -0.02%
Lasso    Sharpe (test): -0.0157  mean monthly: -0.01%
RF       Sharpe (test): 0.1212  mean monthly: 0.09%
XGBoost  Sharpe (test): -0.5964  mean monthly: -0.32%


## Experiment: Market-Adjusted Target + Feature Re-evaluation

### What we did so far

We tried predicting raw 21-day forward returns using various models and feature sets. All models produced negative out-of-sample R² and Sharpe, primarily due to a regime change between the 2010-2022 training period and the 2022-2026 test period.

**Key finding:** Switching the target to **market-adjusted returns** (stock return minus SPY return) dramatically improved results:

| Model | Test Sharpe (raw target) | Test Sharpe (excess target) |
|---|---|---|
| OLS | -0.29 | -0.04 |
| Ridge | — | -0.03 |
| Lasso | — | -0.02 |
| **Random Forest** | -0.18% | **+0.12** |
| XGBoost | — | -0.60 |

Random Forest with market-adjusted returns is our best model so far with a positive out-of-sample Sharpe of 0.12.

### What we are going to do next

Previously we dropped **volatility** and **52-week high ratio** because they didn't improve predictions of raw returns. With a market-adjusted target, these features may behave differently:

- **Volatility** — with market direction removed from the target, stock-level volatility may now capture genuine idiosyncratic risk
- **52-week high ratio** — may recover the expected positive sign when predicting excess returns rather than raw returns

We will add these features back and re-evaluate all models.

In [56]:
# ── Compute analytics for all models ─────────────────────────────────
_ = compute_analytics(X_test_adj, y_test_adj, test_adj['date'], model_adj, 'OLS')
_ = compute_analytics(X_test_adj, y_test_adj, test_adj['date'], ridge_adj, 'Ridge')
_ = compute_analytics(X_test_adj, y_test_adj, test_adj['date'], lasso_adj, 'Lasso')
_ = compute_analytics(X_test_adj, y_test_adj, test_adj['date'], rf_adj, 'Random Forest')
_ = compute_analytics(X_test_adj, y_test_adj, test_adj['date'], xgb_adj, 'XGBoost')

── OLS ──────────────────────────
Sharpe ratio:      -0.0391
Annualized return: -0.29%
Total return:      -1.43%
Win rate:          58.1%
Max drawdown:      -10.65%

── Ridge ──────────────────────────
Sharpe ratio:      -0.0278
Annualized return: -0.20%
Total return:      -1.21%
Win rate:          58.1%
Max drawdown:      -10.43%

── Lasso ──────────────────────────
Sharpe ratio:      -0.0157
Annualized return: -0.12%
Total return:      -1.00%
Win rate:          58.1%
Max drawdown:      -10.49%

── Random Forest ──────────────────────────
Sharpe ratio:      0.1212
Annualized return: 1.07%
Total return:      1.82%
Win rate:          48.4%
Max drawdown:      -7.03%

── XGBoost ──────────────────────────
Sharpe ratio:      -0.5964
Annualized return: -3.84%
Total return:      -9.94%
Win rate:          61.3%
Max drawdown:      -6.60%



In [57]:
# ── Add volatility and 52-week high ratio back to panel_adj ───────────
vol_long = volatility.stack().reset_index()
vol_long.columns = ['date', 'ticker', 'volatility']
panel_adj = panel_adj.merge(vol_long, on=['date', 'ticker'], how='left')
panel_adj['volatility'] = mstats.winsorize(panel_adj['volatility'].fillna(panel_adj['volatility'].median()), limits=[0.01, 0.01])
panel_adj['volatility'] = (panel_adj['volatility'] - panel_adj['volatility'].mean()) / panel_adj['volatility'].std()

ratio_long = ratio_52w.stack().reset_index()
ratio_long.columns = ['date', 'ticker', 'ratio_52w']
panel_adj = panel_adj.merge(ratio_long, on=['date', 'ticker'], how='left')
panel_adj['ratio_52w'] = panel_adj['ratio_52w'].fillna(panel_adj['ratio_52w'].median())
panel_adj['ratio_52w'] = (panel_adj['ratio_52w'] - panel_adj['ratio_52w'].mean()) / panel_adj['ratio_52w'].std()

print(f'Panel shape: {panel_adj.shape}')
print(f'Columns: {list(panel_adj.columns)}')

Panel shape: (1257950, 23)
Columns: ['date', 'ticker', 'forward_return', 'momentum', 'VIX', 'Mkt_RF', 'SMB', 'HML', 'FEDFUNDS', 'CPI_YoY', 'UNRATE', 'reversal', 'mom_3m', 'mom_6m', 'price_accel', 'beta', 'idio_vol_x', 'idio_vol_y', 'idio_vol', 'spy_21d', 'excess_return', 'volatility', 'ratio_52w']


In [58]:
# ── Rebuild train/test with expanded features ─────────────────────────
features_expanded = ['momentum', 'mom_3m', 'mom_6m', 'reversal', 'volatility', 'ratio_52w']

panel_monthly_exp = (
    panel_adj.groupby('ticker', group_keys=False)
    .apply(lambda x: x.iloc[::21])
    .reset_index(drop=True)
)

dates = sorted(panel_monthly_exp['date'].unique())
cutoff = dates[int(len(dates) * 0.8)]

train_exp = panel_monthly_exp[panel_monthly_exp['date'] <= cutoff]
test_exp  = panel_monthly_exp[panel_monthly_exp['date'] > cutoff]

X_train_exp = train_exp[features_expanded]
X_test_exp  = test_exp[features_expanded]
y_train_exp = train_exp['excess_return']
y_test_exp  = test_exp['excess_return']

# ── Train all models ──────────────────────────────────────────────────
model_exp = sm.OLS(y_train_exp, sm.add_constant(X_train_exp)).fit()

rf_exp = RandomForestRegressor(n_estimators=100, max_depth=3, min_samples_leaf=200, max_features=0.5, random_state=42, n_jobs=-1)
rf_exp.fit(X_train_exp, y_train_exp)

xgb_exp = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, min_child_weight=50, random_state=42, n_jobs=-1)
xgb_exp.fit(X_train_exp, y_train_exp)

# ── Analytics ─────────────────────────────────────────────────────────
_ = compute_analytics(X_test_exp, y_test_exp, test_exp['date'], model_exp, 'OLS')
_ = compute_analytics(X_test_exp, y_test_exp, test_exp['date'], rf_exp, 'Random Forest')
_ = compute_analytics(X_test_exp, y_test_exp, test_exp['date'], xgb_exp, 'XGBoost')

── OLS ──────────────────────────
Sharpe ratio:      -1.1094
Annualized return: -9.51%
Total return:      -22.59%
Win rate:          38.7%
Max drawdown:      -25.26%

── Random Forest ──────────────────────────
Sharpe ratio:      0.2632
Annualized return: 2.42%
Total return:      5.33%
Win rate:          64.5%
Max drawdown:      -14.50%

── XGBoost ──────────────────────────
Sharpe ratio:      0.3410
Annualized return: 2.97%
Total return:      6.94%
Win rate:          48.4%
Max drawdown:      -11.66%



### Results — Expanded Feature Set (6 stock-specific features)

Adding volatility and 52-week high ratio back to the market-adjusted target model:

| Model | Sharpe (4 features) | Sharpe (6 features) | Total Return | Win Rate | Max Drawdown |
|---|---|---|---|---|---|
| OLS | -0.04 | -1.11 | -22.59% | 38.7% | -25.26% |
| Random Forest | +0.12 | **+0.26** | +5.33% | 64.5% | -14.50% |
| XGBoost | -0.60 | **+0.34** | +6.94% | 48.4% | -11.66% |

**Key findings:**

- Adding volatility and 52-week high ratio significantly improved non-linear models but hurt OLS badly. This confirms these features have **non-linear relationships** with excess returns that OLS cannot capture.
- **XGBoost is now the best model** with Sharpe of 0.34, total return of +6.94% and lowest max drawdown of -11.66% in the test period.
- The 52-week high ratio now behaves correctly — non-linear models can exploit the reference point effect that was obscured in OLS.
- **Win rate vs Sharpe divergence:** XGBoost wins only 48.4% of months but has the best Sharpe — it wins bigger than it loses. OLS wins 38.7% of months and has the worst Sharpe.

**Next experiment:** Add macro features back to see if non-linear models can exploit them without the regime-specific overfitting that hurt linear models.

### Results — All Features (stock-specific + macro)

Adding macro features (VIX, Mkt_RF, SMB, HML, FEDFUNDS, CPI_YoY, UNRATE) back to the best models:

| Model | Sharpe (6 stock features) | Sharpe (13 all features) | Change |
|---|---|---|---|
| Random Forest | +0.26 | -0.77 | catastrophic |
| XGBoost | +0.34 | -0.93 | catastrophic |

Macro features destroy performance even for non-linear models. The regime change between 2010-2022 and 2022-2026 is too large for any model to bridge.

---

## Final Conclusion

After systematically experimenting with features, targets, and models, the key findings are:

**Best model: XGBoost with 6 stock-specific features + market-adjusted target**
- Sharpe ratio: **0.34**
- Annualized return: **+2.97%**
- Total return: **+6.94%**
- Win rate: **48.4%**
- Max drawdown: **-11.66%**

**What worked:**
1. **Market-adjusted target** — predicting excess returns over SPY removes market direction and focuses on stock selection, dramatically improving out-of-sample generalisation
2. **Stock-specific features only** — macro features caused regime-specific overfitting regardless of model complexity
3. **Non-linear models** — RF and XGBoost captured non-linear relationships between volatility, 52-week high ratio and excess returns that OLS could not

**What didn't work:**
1. **Macro features** — regime change between low-rate (2010-2022) and high-rate (2022-2026) environments broke all macro-based relationships
2. **OLS with non-linear features** — linear models cannot exploit volatility and 52-week high ratio
3. **More complex models with macro features** — complexity amplified overfitting rather than reducing it

**Key lesson:** In quantitative finance, simpler and more regime-agnostic features often outperform complex macro signals out-of-sample. A model that knows *which stocks beat the market* is more robust than one that tries to predict *where the market is going*.

In [60]:
# ── Rolling Window Walk-Forward Backtest ──────────────────────────────
# At each date T: train on trailing 1-year window, predict at T
# Features z-scored cross-sectionally each month

features_roll = ['momentum', 'mom_3m', 'mom_6m', 'reversal', 'volatility', 'ratio_52w']

# Use daily panel (not monthly subsampled) for the rolling window
# Then subsample predictions to monthly
panel_roll = panel_adj[['date', 'ticker', 'excess_return'] + features_roll].dropna()
panel_roll = panel_roll.sort_values('date')

dates = sorted(panel_roll['date'].unique())
window = 252  # 1-year trailing window in trading days

predictions = []

for i, date in enumerate(dates):
    # Get trailing 1-year window of data (strictly before date)
    window_start = dates[max(0, i - window)]
    train_window = panel_roll[(panel_roll['date'] >= window_start) & 
                              (panel_roll['date'] < date)]
    
    if len(train_window) < window * 10:  # require full window
        continue
    
    # Cross-sectional z-score features on training data
    X_train_roll = train_window[features_roll].copy()
    means = X_train_roll.mean()
    stds  = X_train_roll.std()
    X_train_roll = (X_train_roll - means) / stds
    y_train_roll = train_window['excess_return']
    
    # Fit OLS
    model_roll = sm.OLS(y_train_roll, sm.add_constant(X_train_roll)).fit()
    
    # Get test data for date T, z-score cross-sectionally
    test_window = panel_roll[panel_roll['date'] == date].copy()
    X_test_roll = test_window[features_roll].copy()
    X_test_roll = (X_test_roll - means) / stds  # use training means/stds
    
    # Predict
    pred = model_roll.predict(sm.add_constant(X_test_roll))
    test_window = test_window.copy()
    test_window['predicted'] = pred.values
    predictions.append(test_window)

predictions_df = pd.concat(predictions, ignore_index=True)
print(f'Predictions shape: {predictions_df.shape}')
print(f'Date range: {predictions_df["date"].min().date()} to {predictions_df["date"].max().date()}')

Predictions shape: (1255318, 10)
Date range: 2011-02-14 to 2026-04-16


In [61]:
# ── Analytics on rolling walk-forward predictions ─────────────────────
# Subsample to monthly (every 21 days) to avoid overlapping returns
predictions_monthly = (
    predictions_df.groupby('ticker', group_keys=False)
    .apply(lambda x: x.iloc[::21])
    .reset_index(drop=True)
)

# Compute long-short portfolio returns
monthly_returns = []
for date, group in predictions_monthly.groupby('date'):
    if len(group) < 10:
        continue
    q20 = group['predicted'].quantile(0.2)
    q80 = group['predicted'].quantile(0.8)
    long_ret  = group[group['predicted'] >= q80]['excess_return'].mean()
    short_ret = group[group['predicted'] <= q20]['excess_return'].mean()
    monthly_returns.append({'date': date, 'return': long_ret - short_ret})

monthly_returns = pd.DataFrame(monthly_returns).set_index('date')['return']

sharpe     = monthly_returns.mean() / monthly_returns.std() * np.sqrt(12)
ann_return = monthly_returns.mean() * 12
total_ret  = (1 + monthly_returns).prod() - 1
win_rate   = (monthly_returns > 0).mean()
max_dd     = (monthly_returns.cumsum() - monthly_returns.cumsum().cummax()).min()

print(f'── Rolling OLS (1-year window) ──────────────────')
print(f'Sharpe ratio:      {sharpe:.4f}')
print(f'Annualized return: {ann_return*100:.2f}%')
print(f'Total return:      {total_ret*100:.2f}%')
print(f'Win rate:          {win_rate*100:.1f}%')
print(f'Max drawdown:      {max_dd*100:.2f}%')
print(f'Periods:           {len(monthly_returns)}')

── Rolling OLS (1-year window) ──────────────────
Sharpe ratio:      0.8966
Annualized return: 10.82%
Total return:      360.23%
Win rate:          59.3%
Max drawdown:      -17.82%
Periods:           182
